In [21]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

california = fetch_california_housing()

df = pd.DataFrame(california.data, columns=california.feature_names)

df['Target_Price'] = california.target

# Elimino i campioni facili da predire
#df = df[df['Target_Price']<5.0]

print(df.info())

print("--- PRIME 5 RIGHE ---")
print(df.head())

print("\n--- CORRELAZIONE CON IL PREZZO ---")
print(df.corr()['Target_Price'].sort_values(ascending=False))

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   MedInc        20640 non-null  float64
 1   HouseAge      20640 non-null  float64
 2   AveRooms      20640 non-null  float64
 3   AveBedrms     20640 non-null  float64
 4   Population    20640 non-null  float64
 5   AveOccup      20640 non-null  float64
 6   Latitude      20640 non-null  float64
 7   Longitude     20640 non-null  float64
 8   Target_Price  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB
None
--- PRIME 5 RIGHE ---
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547

In [22]:
from sklearn.model_selection import train_test_split

features = df.drop(columns=['Target_Price'])

x_train, x_test, y_train, y_test = train_test_split(features, df['Target_Price'], test_size=0.3, random_state=42)

In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# No tuning
ini_model = RandomForestRegressor(n_estimators=100, max_depth=None, min_samples_leaf=2, max_features='sqrt', bootstrap=True, random_state=42, n_jobs=-1)
ini_model.fit(x_train, y_train)

cv_train = cross_val_score(ini_model, x_train, y_train, cv=5, scoring='r2').mean()
score_test = ini_model.score(x_test, y_test)
print(f"CV train: {cv_train:.4f} | Test: {score_test:.4f}")

CV train: 0.8053 | Test: 0.8086


In [24]:
# With tuning
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 40),
        #'max_depth': trial.suggest_categorical('max_depth', [None, 10, 21, 30, 40, 50]),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5]),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False])        
    }

    model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
    return cross_val_score(model, x_train, y_train, cv=5, scoring='r2').mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'Best params: {study.best_params}')

print(f'Best score: {study.best_value}')

Best trial: 29. Best value: 0.819706: 100%|██████████| 50/50 [04:28<00:00,  5.38s/it]

Best params: {'n_estimators': 427, 'max_depth': 28, 'min_samples_leaf': 1, 'max_features': 'log2', 'bootstrap': False}
Best score: 0.8197064460890242


In [25]:
best_model = RandomForestRegressor(**study.best_params, n_jobs=-1, random_state=42)
best_model.fit(x_train, y_train)
best_model.score(x_test, y_test)

0.8239971598907336